In [5]:
import json
import time
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output


class NPZReplayViewer:
    def __init__(self, npz_path, save_path="annotations.json"):
        self.npz_path = Path(npz_path)
        self.save_path = Path(save_path)

        self.data = np.load(self.npz_path, allow_pickle=True)

        self.positions = self.data["positions"]
        self.goals = self.data["goals"]
        self.obstacles = self.data["obstacles"]

        self.num_steps = self.positions.shape[0]
        self.num_agents = self.positions.shape[1]
        self.num_rows, self.num_cols = self.obstacles.shape

        self.scenario_id = self.npz_path.stem
        self.current_timestep = 0
        self.annotations = []
        self.is_playing = False

        self.agent_colors = plt.cm.tab20(
            np.linspace(0, 1, max(self.num_agents, 1))
        )

        self._load_existing_annotations()
        self._build_widgets()

    def _map_configuration_at_timestep(self):
        return {
            "timestep": self.current_timestep,
            "positions": self.positions[self.current_timestep].astype(int).tolist(),
            "goals": self.goals.astype(int).tolist(),
            "obstacles": self.obstacles.astype(int).tolist(),
            "num_agents": int(self.num_agents),
            "num_rows": int(self.num_rows),
            "num_cols": int(self.num_cols),
        }

    def _load_existing_annotations(self):
        if not self.save_path.exists():
            self.annotations = []
            return

        try:
            with open(self.save_path, "r") as f:
                all_annotations = json.load(f)

            self.annotations = all_annotations.get(self.scenario_id, [])
        except Exception:
            self.annotations = []

    def _save_annotations(self):
        all_annotations = {}

        if self.save_path.exists():
            try:
                with open(self.save_path, "r") as f:
                    all_annotations = json.load(f)
            except Exception:
                all_annotations = {}

        all_annotations[self.scenario_id] = self.annotations

        with open(self.save_path, "w") as f:
            json.dump(all_annotations, f, indent=2)

    def _build_widgets(self):
        self.slider = widgets.IntSlider(
            value=0,
            min=0,
            max=self.num_steps - 1,
            step=1,
            description="Timestep:",
            continuous_update=False,
            layout=widgets.Layout(width="600px"),
        )

        self.prev_button = widgets.Button(description="Prev")
        self.next_button = widgets.Button(description="Next")
        self.play_button = widgets.Button(description="Play")
        self.pause_button = widgets.Button(description="Pause")

        self.play_delay = widgets.FloatSlider(
            value=0.25,
            min=0.05,
            max=2.0,
            step=0.05,
            description="Delay:",
            continuous_update=False,
            layout=widgets.Layout(width="400px"),
        )

        self.mark_button = widgets.Button(description="Mark")
        self.delete_button = widgets.Button(description="Delete Mark")

        self.annotation_type = widgets.Dropdown(
            options=[
                "Deadlock",
                "Congestion",
                "Convention violation",
            ],
            value="Deadlock",
            description="Type:",
        )

        self.notes = widgets.Text(
            value="",
            placeholder="Optional note",
            description="Note:",
            layout=widgets.Layout(width="600px"),
        )

        self.slider.observe(self._on_slider_change, names="value")
        self.prev_button.on_click(self._on_prev_clicked)
        self.next_button.on_click(self._on_next_clicked)
        self.play_button.on_click(self._on_play_clicked)
        self.pause_button.on_click(self._on_pause_clicked)
        self.mark_button.on_click(self._on_mark_clicked)
        self.delete_button.on_click(self._on_delete_clicked)

        self.out = widgets.Output()
        self.info_out = widgets.Output()

        self.controls = widgets.VBox([
            self.slider,
            widgets.HBox([
                self.prev_button,
                self.next_button,
                self.play_button,
                self.pause_button,
            ]),
            self.play_delay,
            self.annotation_type,
            self.notes,
            widgets.HBox([self.mark_button, self.delete_button]),
        ])

    def _on_play_clicked(self, _):
        self.is_playing = True

        while self.is_playing and self.current_timestep < self.num_steps - 1:
            self.slider.value = self.current_timestep + 1
            time.sleep(self.play_delay.value)

        self.is_playing = False

    def _on_pause_clicked(self, _):
        self.is_playing = False

    def _on_mark_clicked(self, _):
        entry = {
            "scenario_id": self.scenario_id,
            "npz_path": str(self.npz_path),
            "annotation_type": self.annotation_type.value,
            "note": self.notes.value,
            "configuration": self._map_configuration_at_timestep(),
        }

        existing_idx = next(
            (
                i for i, ann in enumerate(self.annotations)
                if ann["configuration"]["timestep"] == self.current_timestep
            ),
            None,
        )

        if existing_idx is None:
            self.annotations.append(entry)
        else:
            self.annotations[existing_idx] = entry

        self.annotations = sorted(
            self.annotations,
            key=lambda ann: ann["configuration"]["timestep"],
        )

        self._save_annotations()
        self._render()

    def _on_delete_clicked(self, _):
        self.annotations = [
            ann for ann in self.annotations
            if ann["configuration"]["timestep"] != self.current_timestep
        ]

        self._save_annotations()
        self._render()

    def _render(self):
        with self.out:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(7, 7))
            ax.set_facecolor("#f7f7f7")

            for r in range(self.num_rows):
                for c in range(self.num_cols):
                    color = "#2f2f2f" if self.obstacles[r, c] else "#ffffff"

                    rect = patches.Rectangle(
                        (c, r),
                        1,
                        1,
                        facecolor=color,
                        edgecolor="#d0d0d0",
                        linewidth=0.5,
                    )
                    ax.add_patch(rect)

            for i, (gr, gc) in enumerate(self.goals):
                color = self.agent_colors[i % len(self.agent_colors)]

                goal = patches.Rectangle(
                    (gc + 0.18, gr + 0.18),
                    0.64,
                    0.64,
                    facecolor="none",
                    edgecolor=color,
                    linewidth=2.5,
                    linestyle="--",
                )
                ax.add_patch(goal)

                ax.text(
                    gc + 0.5,
                    gr + 0.5,
                    str(i),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=color,
                    weight="bold",
                )

            positions_t = self.positions[self.current_timestep]

            for i, (r, c) in enumerate(positions_t):
                color = self.agent_colors[i % len(self.agent_colors)]

                circle = patches.Circle(
                    (c + 0.5, r + 0.5),
                    0.33,
                    facecolor=color,
                    edgecolor="black",
                    linewidth=1.2,
                    alpha=0.95,
                )
                ax.add_patch(circle)

                ax.text(
                    c + 0.5,
                    r + 0.5,
                    str(i),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white",
                    weight="bold",
                )

            current_annotation = next(
                (
                    ann for ann in self.annotations
                    if ann["configuration"]["timestep"] == self.current_timestep
                ),
                None,
            )

            title = f"{self.npz_path.name} | timestep {self.current_timestep}/{self.num_steps - 1}"

            if current_annotation:
                title += f" | MARKED: {current_annotation['annotation_type']}"

            ax.set_title(title, fontsize=12, weight="bold", pad=12)

            ax.set_xlim(0, self.num_cols)
            ax.set_ylim(self.num_rows, 0)
            ax.set_aspect("equal")

            ax.set_xticks(np.arange(0, self.num_cols + 1, 1))
            ax.set_yticks(np.arange(0, self.num_rows + 1, 1))
            ax.set_xticklabels([])
            ax.set_yticklabels([])
            ax.tick_params(length=0)

            for spine in ax.spines.values():
                spine.set_visible(False)

            plt.tight_layout()
            plt.show()

        self._render_info()

    def _render_info(self):
        with self.info_out:
            clear_output(wait=True)

            current_annotation = next(
                (
                    ann for ann in self.annotations
                    if ann["configuration"]["timestep"] == self.current_timestep
                ),
                None,
            )

            print(f"Scenario: {self.scenario_id}")
            print(f"Current timestep: {self.current_timestep}")

            if current_annotation:
                print(f"Current annotation: {current_annotation['annotation_type']}")
                print(f"Note: {current_annotation['note']}")
            else:
                print("Current annotation: None")

            print("\nAll annotations:")
            if not self.annotations:
                print("None")
            else:
                for ann in self.annotations:
                    t = ann["configuration"]["timestep"]
                    label = ann["annotation_type"]
                    note = ann["note"]
                    print(f"- t={t}: {label} | {note}")

            print(f"\nSaved to: {self.save_path}")

    def _on_slider_change(self, change):
        self.current_timestep = change["new"]
        self._render()

    def _on_prev_clicked(self, _):
        self.is_playing = False

        if self.current_timestep > 0:
            self.slider.value = self.current_timestep - 1

    def _on_next_clicked(self, _):
        self.is_playing = False

        if self.current_timestep < self.num_steps - 1:
            self.slider.value = self.current_timestep + 1

    def show(self):
        display(self.controls)
        display(self.out)
        display(self.info_out)
        self._render()

In [6]:
viewer = NPZReplayViewer("dataset/held_out/ckpt_30000/random_ms147_ss1002_na48.npz")
viewer.show()

Output()

Output()